In [1]:
import fastf1
import pandas as pd

# 1. Mount your local data folder to fastf1's cache
# This ensures you are loading the raw race data you already have stored locally
fastf1.Cache.enable_cache('../data/raw')

# 2. Load the 2025 Australian Grand Prix (Round 1), Race Session ('R')
# The 'Race' session is essential for capturing prolonged tyre degradation data
session = fastf1.get_session(2025, 1, 'R')
session.load()
# 1. Extract the Laps DataFrame
laps = session.laps

# 2. Goal 4: Data Filtering (Noise Reduction)
# Create a clean copy to avoid Pandas SettingWithCopyWarnings
clean_laps = laps.copy()

# Filter 1: Drop inherently inaccurate laps (In-laps, Out-laps, and massively anomalous laps)
clean_laps = clean_laps[clean_laps['IsAccurate'] == True]



# Filter 2: Explicitly drop any laps involving the pit lane (Belt and suspenders approach)
clean_laps = clean_laps[clean_laps['PitInTime'].isnull() & clean_laps['PitOutTime'].isnull()]


# Filter 3: Drop Safety Car (SC) and Virtual Safety Car (VSC) laps
# TrackStatus '4' is SC, '6' is VSC. We drop any lap where these string flags appear.
clean_laps = clean_laps[~clean_laps['TrackStatus'].str.contains('4|6', na=False)]



# 3. Goal 5: Create the Stint_ID (Cross-Validation Grouping Key)
# A 'Stint' is a continuous run on one set of tires. 
# We combine the Driver's 3-letter code with their integer Stint number.
clean_laps['Stint_ID'] = clean_laps['Driver'] + "_Stint_" + clean_laps['Stint'].astype(int).astype(str)



# 4. Goal 6: Define Target and Calculate the Pace Baseline
# Define the target variable (What we want to predict)
clean_laps['Target_Compound'] = clean_laps['Compound']



# Convert timedelta LapTime to workable float seconds
clean_laps['LapTime_Sec'] = clean_laps['LapTime'].dt.total_seconds()

# Calculate the Pace Baseline: The fastest lap time recorded on THAT SPECIFIC LapNumber across all drivers
field_best_per_lap = clean_laps.groupby('LapNumber')['LapTime_Sec'].min().reset_index()
field_best_per_lap.rename(columns={'LapTime_Sec': 'Field_Best_Lap_Sec'}, inplace=True)

# Merge the baseline back into our clean_laps dataframe
clean_laps = clean_laps.merge(field_best_per_lap, on='LapNumber', how='left')

# Calculate the Delta: How much slower was this driver compared to the fastest car on this exact lap?
clean_laps['Delta_to_Field_Best'] = clean_laps['LapTime_Sec'] - clean_laps['Field_Best_Lap_Sec']

# Drop columns that are no longer needed or could cause data leakage
# cols_to_keep = [
#     'Driver', 'LapNumber', 'Stint_ID', 'Target_Compound', 'TyreLife', 
#     'LapTime_Sec', 'Field_Best_Lap_Sec', 'Delta_to_Field_Best', 'TrackStatus'
# ]
# clean_laps = clean_laps[cols_to_keep]

print(f"Original laps: {len(laps)} | Clean laps remaining: {len(clean_laps)}")
print(clean_laps.head())

import numpy as np

# --- PHASE 3: TELEMETRY AGGREGATION & PHYSICS PROXIES ---

print("Starting telemetry aggregation. This may take a few minutes...")

# 1. Initialize lists to store our aggregated lap data
telemetry_features = []

# Loop through every clean lap we filtered in Phase 2
for index, lap in clean_laps.iterrows():
    # Fetch the high-frequency telemetry for this specific lap
    try:
        tel = lap.get_telemetry()
    except Exception as e:
        # Occasionally a lap's telemetry is corrupted; we skip it
        continue 
    
    # Calculate our basic aggregations
    avg_speed = tel['Speed'].mean()
    std_dev_speed = tel['Speed'].std()
    avg_rpm = tel['RPM'].mean()
    avg_throttle = tel['Throttle'].mean()
    
    # Percent full throttle (Throttle is 100% or very close to it)
    percent_full_throttle = (tel['Throttle'] >= 99).mean()
    
    # Brake Events (Count how many times the brake pedal goes from 0 to >0)
    # We use diff() to find the transitions
    brake_transitions = (tel['Brake'] > 0).astype(int).diff()
    brake_events = (brake_transitions == 1).sum()
    
    # Dirty Air / Traffic Proxy (FastF1 DRS values >= 10 typically indicate the flap is open)
    drs_active = int((tel['DRS'] >= 10).any())
    
    # Append the extracted features alongside the original DataFrame's index 
    # so we can merge it back perfectly
    telemetry_features.append({
        'Index': index, # Keep the index for merging
        'Avg_Speed': avg_speed,
        'Std_Dev_Speed': std_dev_speed,
        'Avg_RPM': avg_rpm,
        'Avg_Throttle': avg_throttle,
        'Percent_Full_Throttle': percent_full_throttle,
        'Brake_Events': brake_events,
        'DRS_Active': drs_active
    })

# Convert the list of dictionaries into a DataFrame
tel_df = pd.DataFrame(telemetry_features)
tel_df.set_index('Index', inplace=True)

# 2. Merge telemetry features back into our clean_laps DataFrame
clean_laps = clean_laps.join(tel_df)

# Drop any rows where telemetry was missing
clean_laps.dropna(subset=['Avg_Speed'], inplace=True)

# 3. Add Physics Proxies
# Simple Fuel Mass Proxy (Linear burn)
clean_laps['Estimated_Fuel_Mass'] = 110.0 - (clean_laps['LapNumber'] * 1.5)

# Degradation Derivative (3-lap rolling average of the pace delta)
# We MUST group by Stint_ID so the rolling average doesn't bleed across pit stops!
clean_laps['Degradation_Slope'] = (
    clean_laps.groupby('Stint_ID')['Delta_to_Field_Best']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

# 4. Merge Weather Data (Track Temperature)
# Extract weather data from the session
weather_data = session.weather_data

# Weather data needs to be sorted by time for an 'asof' merge
weather_data = weather_data.sort_values(by='Time')
clean_laps = clean_laps.sort_values(by='LapStartTime')

weather_subset = weather_data[['Time', 'TrackTemp', 'AirTemp']].rename(columns={'Time': 'Weather_Time'})

# Perform the asof merge: For every lap, find the most recent weather recorded BEFORE the lap started
clean_laps = pd.merge_asof(
    clean_laps, 
    weather_subset,
    weather_data[['Time', 'TrackTemp', 'AirTemp']], 
    left_on='LapStartTime', 
    right_on='Weather_Time', 
    direction='backward'
)

# Clean up final dataframe and drop raw timestamps to prevent model confusion
clean_laps.drop(columns=['Weather_Time'], inplace=True)

print(f"Feature engineering complete. Final shape: {clean_laps.shape}")
print(clean_laps[['Driver', 'LapNumber', 'Estimated_Fuel_Mass', 'Degradation_Slope', 'TrackTemp', 'Avg_Speed']].head())

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core    

Original laps: 927 | Clean laps remaining: 568
                    Time Driver DriverNumber                LapTime  \
0 0 days 01:29:43.766000    VER            1 0 days 00:01:36.830000   
1 0 days 01:31:18.081000    VER            1 0 days 00:01:34.315000   
2 0 days 01:32:50.731000    VER            1 0 days 00:01:32.650000   
3 0 days 01:34:23.184000    VER            1 0 days 00:01:32.453000   
4 0 days 01:35:54.775000    VER            1 0 days 00:01:31.591000   

   LapNumber  Stint PitOutTime PitInTime            Sector1Time  \
0        8.0    4.0        NaT       NaT 0 days 00:00:35.249000   
1        9.0    4.0        NaT       NaT 0 days 00:00:33.850000   
2       10.0    4.0        NaT       NaT 0 days 00:00:33.117000   
3       11.0    4.0        NaT       NaT 0 days 00:00:33.017000   
4       12.0    4.0        NaT       NaT 0 days 00:00:32.767000   

             Sector2Time  ... Position Deleted DeletedReason FastF1Generated  \
0 0 days 00:00:20.145000  ...      2.0   Fa

MergeError: Can only pass argument "on" OR "left_on" and "right_on", not a combination of both.